<center> <h1> Generating Poetic Texts using RNN </h1> </center>

In [30]:
#Import required libraries:
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input
from tensorflow.keras.optimizers import RMSprop



In [31]:
#Get the Shakespeare text:
#https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt
filepath = tf.keras.utils.get_file("shakespeare.txt","https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt")


In [32]:
#Get the text of this file into the script:
text = open(filepath,"rb").read().decode(encoding="utf-8").lower()

In [33]:
#Convert this text into a numerical-format (so that our neural-network is able to properly deal with the text):
#We will pass using the format of numpy array for our neural-network:

#We will only use a part of text (so that our neural-network doesn't take too much time on training):
text = text[300000:800000]

#Get a set of possible characters from the text:
characters = sorted(set(text))

#Create 2 dictionaries so that we can convert these characters from characters-to-numeric and back from numeric-to-character format:
#NOTE: The numerical values will represent the frequencies of each character.
char_to_index = dict((c,i) for i, c in enumerate(characters))
index_to_char = dict((i,c) for i, c in enumerate(characters))


In [34]:
#Get the feature-data and the target-data:
SEQ_LENGTH = 40
STEP_SIZE = 3

sentences = []  #For loading the sentence
next_characters = [] #for predicting the next character of the sentence
#for-example: 'how are yo'---->'u'--------------->'how are you'

for i in range(0, len(text)-SEQ_LENGTH,STEP_SIZE):
    sentences.append(text[i:i+SEQ_LENGTH])
    next_characters.append(text[i+SEQ_LENGTH])

In [35]:
#Convert the training-data into a numpy array:
x = np.zeros((len(sentences), SEQ_LENGTH, len(characters)), dtype=bool)
y = np.zeros((len(sentences), len(characters)), dtype=bool)

In [36]:
#Now fill these arrays:
for i , sentence in enumerate(sentences):
    for t , character in enumerate(sentence):
        x[i,t,char_to_index[character]] = 1
    y[i,char_to_index[next_characters[i]]] = 1
        

In [37]:
#Now build the neural-network:
model = Sequential()
model.add(Input(shape=(SEQ_LENGTH,len(characters))))
model.add(LSTM(128))
model.add(Dense(len(characters)))
model.add(Activation('softmax')) #activation layer with activation-function as SOFTMAX

#Compile the model along with the loss-function:
model.compile(loss="categorical_crossentropy",optimizer=RMSprop(learning_rate=0.01))

#Now fit the training-data:
model.fit(x,y,batch_size=256,epochs=4)

#Save the model:
model.save("textgenerator.h5")

Epoch 1/4
651/651 ━━━━━━━━━━━━━━━━━━━━ 135s 204ms/step - loss: 2.1789
Epoch 2/4
651/651 ━━━━━━━━━━━━━━━━━━━━ 131s 202ms/step - loss: 1.7391
Epoch 3/4
651/651 ━━━━━━━━━━━━━━━━━━━━ 132s 203ms/step - loss: 1.6068
Epoch 4/4
651/651 ━━━━━━━━━━━━━━━━━━━━ 131s 201ms/step - loss: 1.5337


In [38]:
'''
For loading the model:

from tensorflow.keras.models import load_model
model = load_model("textgenerator.keras")
'''

'\nFor loading the model:\n\nfrom tensorflow.keras.models import load_model\nmodel = load_model("textgenerator.keras")\n'

In [39]:
#Function to predict the next character in the sentence:
def sample(preds, temperature=1.0):
    preds = np.asarray(preds).astype("float64")
    preds = np.log(preds) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

#Function which genrates text:
def generate_text(length,temperature):
    start_index = random.randint(0,len(text)-SEQ_LENGTH - 1)
    generated = ""
    sentence = text[start_index:start_index+SEQ_LENGTH]
    generated += sentence

    for i in range(length):
        x = np.zeros((1,SEQ_LENGTH,len(characters)))
        for t , character in enumerate(sentence):
            x[0,t,char_to_index[character]] = 1

        predictions = model.predict(x,verbose=0)[0]
        next_index = sample(predictions,temperature)
        next_character = index_to_char[next_index]

        generated += next_character
        sentence = sentence[1:] + next_character
        
    return generated

In [40]:
print("\n\n-------Temperature as 0.2-------------")
print(generate_text(300,0.2))
print("\n\n-------Temperature as 0.4-------------")
print(generate_text(300,0.4))
print("\n\n-------Temperature as 0.6-------------")
print(generate_text(300,0.6))
print("\n\n-------Temperature as 0.8-------------")
print(generate_text(300,0.8))
print("\n\n-------Temperature as 1-------------")
print(generate_text(300,1))



-------Temperature as 0.2-------------
where now we mean to stand.

second keeper:
the truth and many the cannot son i have in the son in the death.

second witchilings:
if you have the lanch the lanch and the counders
to the battle the crown and the king here with the beath,
i have the bears and the lanch the canst the dead;
and in the soul is the canst the come of the brothe


-------Temperature as 0.4-------------
 do it?

queen margaret:
ay, but thou use the summer the have in the brother heart:
where is the bears is shall be to but the trainor
to stand and speak of my lord of the counders,
i cannot the proposon to be our speak to the must.

nurse:
these propost the world a bear the see me son it is not say.

king richard ii:
why, which the burded


-------Temperature as 0.6-------------
en neptune
a ram, and bleated; and the from that hearth.

rich of york:
why, hears of the battle be nuch is the blood,
and a why son i know to the brother as gone?

duke of guester:
and maday, an